In [1]:
import pandas as pd

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

psych_path = r"C:\Users\medwards\OneDrive - HDR, Inc\Arch. Advisory Services - Clients\Virginia\Riverside\mwe.01\data\runs\run_20260611_142421\outputs\2025_Psych_acuity_rmc_emergency_ts.rollup.csv"

nonpsych_path = r"C:\Users\medwards\OneDrive - HDR, Inc\Arch. Advisory Services - Clients\Virginia\Riverside\mwe.01\data\runs\run_20260611_142421\outputs\2025_Non-Psych_acuity_rmc_emergency_ts.rollup.csv"

# 🔧 Separate thresholds (YOU control these)
PSYCH_THRESHOLD = 2
NONPSYCH_THRESHOLD = 31


# --------------------------------------------------
# FUNCTION
# --------------------------------------------------

def analyze_file(path, threshold, label):

    print("\n" + "="*60)
    print(f"Analyzing: {label}")
    print(f"File: {path}")
    print("="*60)

    df = pd.read_csv(path)

    # -----------------------------
    # Validate columns
    # -----------------------------
    required_cols = ["interval", "census"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"{label}: Missing column '{col}'")

    # -----------------------------
    # Clean types
    # -----------------------------
    df["interval"] = pd.to_datetime(df["interval"], errors="coerce")
    df["census"] = pd.to_numeric(df["census"], errors="coerce")

    df = df.dropna(subset=["interval", "census"])

    # -----------------------------
    # Metric
    # -----------------------------
    df["above_threshold"] = df["census"] > threshold

    total_minutes = len(df)
    minutes_above = df["above_threshold"].sum()

    pct_above = minutes_above / total_minutes if total_minutes > 0 else 0

    # -----------------------------
    # Output
    # -----------------------------
    print("\n--- RESULTS ---")
    print(f"Threshold: {threshold}")
    print(f"Total minutes: {total_minutes:,}")
    print(f"Minutes above: {minutes_above:,}")
    print(f"Pct time above: {pct_above:.4%}")

    return {
        "group": label,
        "threshold": threshold,
        "total_minutes": total_minutes,
        "minutes_above": minutes_above,
        "pct_above": pct_above
    }


# --------------------------------------------------
# MAIN
# --------------------------------------------------

if __name__ == "__main__":

    results = []

    results.append(analyze_file(psych_path, PSYCH_THRESHOLD, "Psych"))
    results.append(analyze_file(nonpsych_path, NONPSYCH_THRESHOLD, "Non-Psych"))

    # -----------------------------
    # SUMMARY TABLE
    # -----------------------------
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)

    summary_df = pd.DataFrame(results)
    print(summary_df)

    # -----------------------------
    # COMBINED METRIC (OPTIONAL)
    # -----------------------------
    total_minutes = sum(r["total_minutes"] for r in results)
    total_above = sum(r["minutes_above"] for r in results)

    combined_pct = total_above / total_minutes if total_minutes > 0 else 0

    print("\nOverall % time above threshold (combined):")
    print(f"{combined_pct:.4%}")


Analyzing: Psych
File: C:\Users\medwards\OneDrive - HDR, Inc\Arch. Advisory Services - Clients\Virginia\Riverside\mwe.01\data\runs\run_20260611_142421\outputs\2025_Psych_acuity_rmc_emergency_ts.rollup.csv

--- RESULTS ---
Threshold: 2
Total minutes: 525,600
Minutes above: 182,578
Pct time above: 34.7371%

Analyzing: Non-Psych
File: C:\Users\medwards\OneDrive - HDR, Inc\Arch. Advisory Services - Clients\Virginia\Riverside\mwe.01\data\runs\run_20260611_142421\outputs\2025_Non-Psych_acuity_rmc_emergency_ts.rollup.csv

--- RESULTS ---
Threshold: 31
Total minutes: 525,600
Minutes above: 354,679
Pct time above: 67.4808%

SUMMARY
       group  threshold  total_minutes  minutes_above  pct_above
0      Psych          2         525600         182578   0.347371
1  Non-Psych         31         525600         354679   0.674808

Overall % time above threshold (combined):
51.1089%
